# FonoKids -- Entrenamiento de modelos (video + audio) en Colab

Este notebook hace EXACTAMENTE lo mismo que `scripts/train_landmarks_transformer.py` y `scripts/train_audio_classifier.py` de tu PC, pero corriendo en la GPU gratuita de Colab y leyendo/escribiendo desde tu Google Drive.

- **Video**: entrena el modelo TCN + Conformer sobre los `.npy` de landmarks (ya extraidos con el notebook `Entrenar_FonoKids.ipynb` de esta misma carpeta) -> guarda `models/best.pth`.
- **Audio**: entrena el clasificador sobre los embeddings de wav2vec2 (ya extraidos, o los extrae al vuelo si falta alguno) -> guarda `models/audio_classifier.pth`.

Ambos muestran al terminar: **Train Loss vs Val Loss**, **Train Accuracy vs Val Accuracy** y **matriz de confusion**.

### Antes de correr esto

1. Necesitas haber corrido antes el notebook de extraccion (`Entrenar_FonoKids.ipynb`), asi ya tenes en Drive `data/landmarks_npy/` (con `label_map.json`) y `data/audio_embeddings_cache/`.
2. Activa GPU: **Entorno de ejecucion > Cambiar tipo de entorno de ejecucion > GPU**.
3. Corre las celdas de arriba hacia abajo.

## 1. Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Instalar dependencias

In [ ]:
!pip install -q transformers huggingface_hub
import torch
print('CUDA disponible:', torch.cuda.is_available())

## 3. Configuracion -- rutas en Drive

In [ ]:
import os

# Misma carpeta base que usaste en el notebook de extraccion (donde estan
# data/landmarks_npy/ y data/audio_embeddings_cache/).
DRIVE_BASE_DIR = '/content/drive/MyDrive/FonoKids'

RUTA_CLIPS = os.path.join(DRIVE_BASE_DIR, 'data', 'sesiones_continuas')
LANDMARKS_DIR = os.path.join(DRIVE_BASE_DIR, 'data', 'landmarks_npy')
AUDIO_EMB_DIR = os.path.join(DRIVE_BASE_DIR, 'data', 'audio_embeddings_cache')
MODELS_DIR = os.path.join(DRIVE_BASE_DIR, 'models')
OUTPUTS_DIR = os.path.join(DRIVE_BASE_DIR, 'outputs')
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(OUTPUTS_DIR, exist_ok=True)

assert os.path.isdir(LANDMARKS_DIR), f'No encontre {LANDMARKS_DIR} -- corre primero el notebook de extraccion.'
print('Landmarks en:', LANDMARKS_DIR)
print('Embeddings de audio en:', AUDIO_EMB_DIR)
print('Modelos van a guardarse en:', MODELS_DIR)

---
# PARTE A -- Entrenar el modelo de VIDEO (TCN + Conformer)

## 4. Dataset -- lee los .npy de landmarks (identico a lip_reading_dataset.py)

In [ ]:
import glob
import json
import unicodedata
import numpy as np
from torch.utils.data import Dataset, DataLoader


def _nfc(text):
    return unicodedata.normalize('NFC', text)


class LipReadingDataset(Dataset):
    def __init__(self, data_dir, max_frames=60, file_paths=None, augment=False):
        self.data_dir = data_dir
        self.max_frames = max_frames
        self.augment = augment

        label_map_path = os.path.join(data_dir, 'label_map.json')
        if not os.path.exists(label_map_path):
            raise FileNotFoundError(f'No encontre {label_map_path}. Corre el notebook de extraccion primero.')
        with open(label_map_path, 'r', encoding='utf-8') as f:
            raw_label_map = json.load(f)
        self.label_map = {_nfc(word): idx for word, idx in raw_label_map.items()}

        num_classes = max(self.label_map.values()) + 1
        self.class_names = ['?'] * num_classes
        for word, idx in self.label_map.items():
            self.class_names[idx] = word

        if file_paths is not None:
            self.file_paths = list(file_paths)
        else:
            self.file_paths = sorted(glob.glob(os.path.join(data_dir, '*', '*.npy')))
        if not self.file_paths:
            raise RuntimeError(f'No hay archivos .npy en {data_dir}.')

        first_sample = np.load(self.file_paths[0])
        self.landmark_dim = int(first_sample.shape[-1])

    def __len__(self):
        return len(self.file_paths)

    def _label_from_path(self, path):
        word = _nfc(os.path.basename(os.path.dirname(path)))
        return self.label_map[word]

    def _apply_augmentation(self, sequence):
        sequence = sequence + np.random.normal(0, 0.003, sequence.shape).astype(np.float32)
        shift = np.random.uniform(-0.01, 0.01, size=(1, sequence.shape[1])).astype(np.float32)
        scale = np.random.uniform(0.97, 1.03)
        sequence = sequence * scale + shift
        T = sequence.shape[0]
        n_mask = max(1, int(round(T * np.random.uniform(0.08, 0.12))))
        n_mask = min(n_mask, T)
        mask_idx = np.random.choice(T, size=n_mask, replace=False)
        sequence[mask_idx] = 0.0
        return sequence.astype(np.float32)

    def __getitem__(self, idx):
        path = self.file_paths[idx]
        sequence = np.load(path).astype(np.float32)
        label = self._label_from_path(path)
        if self.augment:
            sequence = self._apply_augmentation(sequence)
        T = sequence.shape[0]
        if T < self.max_frames:
            pad = np.zeros((self.max_frames - T, sequence.shape[1]), dtype=np.float32)
            sequence = np.concatenate([sequence, pad], axis=0)
            real_length = T
        elif T > self.max_frames:
            sequence = sequence[: self.max_frames]
            real_length = self.max_frames
        else:
            real_length = T
        return (
            torch.from_numpy(sequence),
            torch.tensor(label, dtype=torch.long),
            torch.tensor(real_length, dtype=torch.long),
        )


def build_dataloaders(data_dir, max_frames=60, batch_size=32, val_ratio=0.2, seed=42, augment_train=True):
    reference = LipReadingDataset(data_dir=data_dir, max_frames=max_frames)
    file_paths = reference.file_paths

    labels_by_path = {p: reference.label_map[_nfc(os.path.basename(os.path.dirname(p)))] for p in file_paths}
    paths_by_class = {}
    for p in file_paths:
        paths_by_class.setdefault(labels_by_path[p], []).append(p)

    rng = np.random.default_rng(seed)
    train_paths, val_paths = [], []
    for label, paths in paths_by_class.items():
        paths = list(paths)
        rng.shuffle(paths)
        n_val = max(1, int(round(len(paths) * val_ratio)))
        val_paths.extend(paths[:n_val])
        train_paths.extend(paths[n_val:])
    rng.shuffle(train_paths)
    rng.shuffle(val_paths)

    train_ds = LipReadingDataset(data_dir=data_dir, max_frames=max_frames, file_paths=train_paths, augment=augment_train)
    val_ds = LipReadingDataset(data_dir=data_dir, max_frames=max_frames, file_paths=val_paths, augment=False)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0, drop_last=False)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0, drop_last=False)

    return reference, train_loader, val_loader

## 5. Modelo TCN + Conformer (identico a train_landmarks_transformer.py)

In [ ]:
import math
import time
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, : x.size(1)]


class TCNBlock(nn.Module):
    def __init__(self, channels, kernel_size=3, dilation=1, dropout=0.1):
        super().__init__()
        padding = (kernel_size - 1) * dilation // 2
        self.conv1 = nn.Conv1d(channels, channels, kernel_size, padding=padding, dilation=dilation)
        self.bn1 = nn.BatchNorm1d(channels)
        self.conv2 = nn.Conv1d(channels, channels, kernel_size, padding=padding, dilation=dilation)
        self.bn2 = nn.BatchNorm1d(channels)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, key_padding_mask=None):
        residual = x
        x = x.transpose(1, 2)
        if key_padding_mask is not None:
            x = x.masked_fill(key_padding_mask.unsqueeze(1), 0.0)
        x = self.dropout(self.relu(self.bn1(self.conv1(x))))
        x = self.bn2(self.conv2(x))
        x = x.transpose(1, 2)
        return self.relu(residual + self.dropout(x))


class TCNFeatureExtractor(nn.Module):
    def __init__(self, channels, n_blocks=3, kernel_size=3, dropout=0.1):
        super().__init__()
        self.blocks = nn.ModuleList([
            TCNBlock(channels, kernel_size=kernel_size, dilation=2 ** i, dropout=dropout)
            for i in range(n_blocks)
        ])

    def forward(self, x, key_padding_mask=None):
        for block in self.blocks:
            x = block(x, key_padding_mask=key_padding_mask)
        return x


class ConformerFeedForward(nn.Module):
    def __init__(self, d_model, dim_feedforward, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, dim_feedforward),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class ConformerConvModule(nn.Module):
    def __init__(self, d_model, kernel_size=15, dropout=0.1):
        super().__init__()
        self.layer_norm = nn.LayerNorm(d_model)
        self.pointwise_conv1 = nn.Conv1d(d_model, 2 * d_model, kernel_size=1)
        self.glu = nn.GLU(dim=1)
        padding = (kernel_size - 1) // 2
        self.depthwise_conv = nn.Conv1d(d_model, d_model, kernel_size=kernel_size, padding=padding, groups=d_model)
        self.batch_norm = nn.BatchNorm1d(d_model)
        self.swish = nn.SiLU()
        self.pointwise_conv2 = nn.Conv1d(d_model, d_model, kernel_size=1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, key_padding_mask=None):
        residual = x
        x = self.layer_norm(x)
        x = x.transpose(1, 2)
        if key_padding_mask is not None:
            x = x.masked_fill(key_padding_mask.unsqueeze(1), 0.0)
        x = self.pointwise_conv1(x)
        x = self.glu(x)
        x = self.depthwise_conv(x)
        x = self.batch_norm(x)
        x = self.swish(x)
        x = self.pointwise_conv2(x)
        x = self.dropout(x)
        x = x.transpose(1, 2)
        return residual + x


class ConformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, dim_feedforward, conv_kernel_size=15, dropout=0.1):
        super().__init__()
        self.ff1 = ConformerFeedForward(d_model, dim_feedforward, dropout)
        self.self_attn_norm = nn.LayerNorm(d_model)
        self.self_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.attn_dropout = nn.Dropout(dropout)
        self.conv_module = ConformerConvModule(d_model, conv_kernel_size, dropout)
        self.ff2 = ConformerFeedForward(d_model, dim_feedforward, dropout)
        self.final_norm = nn.LayerNorm(d_model)

    def forward(self, x, src_key_padding_mask=None):
        x = x + 0.5 * self.ff1(x)
        residual = x
        x_norm = self.self_attn_norm(x)
        attn_out, _ = self.self_attn(x_norm, x_norm, x_norm, key_padding_mask=src_key_padding_mask, need_weights=False)
        x = residual + self.attn_dropout(attn_out)
        x = self.conv_module(x, key_padding_mask=src_key_padding_mask)
        x = x + 0.5 * self.ff2(x)
        return self.final_norm(x)


class ConformerEncoder(nn.Module):
    def __init__(self, d_model, n_layers, n_heads, dim_feedforward, conv_kernel_size=15, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            ConformerBlock(d_model, n_heads, dim_feedforward, conv_kernel_size, dropout)
            for _ in range(n_layers)
        ])

    def forward(self, x, src_key_padding_mask=None):
        for layer in self.layers:
            x = layer(x, src_key_padding_mask=src_key_padding_mask)
        return x


class LipReadingConformer(nn.Module):
    def __init__(self, num_classes, input_dim, hidden_dim=128, n_layers=2, n_heads=8,
                 dim_feedforward=256, conv_kernel_size=15, tcn_blocks=3, tcn_kernel_size=3, dropout=0.3):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, hidden_dim)
        self.tcn = TCNFeatureExtractor(hidden_dim, n_blocks=tcn_blocks, kernel_size=tcn_kernel_size, dropout=dropout)
        self.pos_encoder = PositionalEncoding(hidden_dim)
        self.conformer_encoder = ConformerEncoder(
            d_model=hidden_dim, n_layers=n_layers, n_heads=n_heads,
            dim_feedforward=dim_feedforward, conv_kernel_size=conv_kernel_size, dropout=dropout,
        )
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 64), nn.ReLU(), nn.Dropout(dropout), nn.Linear(64, num_classes),
        )

    def forward_features(self, x, src_key_padding_mask=None):
        x = self.input_proj(x)
        x = self.tcn(x, key_padding_mask=src_key_padding_mask)
        x = self.pos_encoder(x)
        x = self.conformer_encoder(x, src_key_padding_mask=src_key_padding_mask)
        return masked_mean_pool(x, src_key_padding_mask)

    def forward(self, x, src_key_padding_mask=None):
        pooled = self.forward_features(x, src_key_padding_mask=src_key_padding_mask)
        return self.classifier(pooled)


def masked_mean_pool(x, src_key_padding_mask):
    if src_key_padding_mask is None:
        return x.mean(dim=1)
    real_mask = (~src_key_padding_mask).unsqueeze(-1).float()
    summed = (x * real_mask).sum(dim=1)
    counts = real_mask.sum(dim=1).clamp(min=1.0)
    return summed / counts


def make_padding_mask(lengths, max_len, device):
    idx = torch.arange(max_len, device=device).unsqueeze(0)
    return idx >= lengths.unsqueeze(1)


class EarlyStopping:
    def __init__(self, patience=8, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None or val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True


def run_epoch(model, loader, criterion, optimizer, device, train):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for sequences, labels, lengths in loader:
            sequences = sequences.to(device)
            labels = labels.to(device)
            lengths = lengths.to(device)
            mask = make_padding_mask(lengths, sequences.shape[1], device)
            if train:
                optimizer.zero_grad()
            outputs = model(sequences, src_key_padding_mask=mask)
            loss = criterion(outputs, labels)
            if train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
            total_loss += loss.item() * sequences.size(0)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)
    avg_loss = total_loss / total if total > 0 else 0.0
    acc = 100.0 * correct / total if total > 0 else 0.0
    return avg_loss, acc


def build_warmup_cosine_scheduler(optimizer, epochs, warmup_epochs=5):
    warmup_epochs = max(1, min(warmup_epochs, epochs - 1)) if epochs > 1 else 1

    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        progress = (epoch - warmup_epochs) / max(1, epochs - warmup_epochs)
        return 0.5 * (1 + math.cos(math.pi * progress))

    return optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

## 6. Entrenar el modelo de video

Guarda `models/best.pth` y muestra las graficas de Loss/Accuracy + matriz de confusion al terminar (igual que en tu PC).

In [ ]:
VIDEO_MODEL_PATH = os.path.join(MODELS_DIR, 'best.pth')
SEED = 42
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Entrenando en: {device}')

EPOCHS = 50
BATCH_SIZE = 32
MAX_FRAMES = 60
LR = 1e-3
WEIGHT_DECAY = 1e-3
VAL_RATIO = 0.2
EARLY_STOPPING_PATIENCE = 15
WARMUP_EPOCHS = 5
LABEL_SMOOTHING = 0.1

dataset, train_loader, val_loader = build_dataloaders(
    data_dir=LANDMARKS_DIR, max_frames=MAX_FRAMES, batch_size=BATCH_SIZE,
    val_ratio=VAL_RATIO, seed=SEED, augment_train=True,
)
num_classes = len(dataset.class_names)
input_dim = dataset.landmark_dim
print(f'Clases ({num_classes}): {dataset.class_names}')
print(f'Dimension de entrada detectada: {input_dim}')
print(f'Total muestras: {len(dataset)} | Train: {len(train_loader.dataset)} | Val: {len(val_loader.dataset)}')

model = LipReadingConformer(num_classes=num_classes, input_dim=input_dim).to(device)

train_labels = [dataset._label_from_path(p) for p in train_loader.dataset.file_paths]
class_counts = np.bincount(train_labels, minlength=num_classes).astype(np.float64)
class_counts[class_counts == 0] = 1.0
class_weights = class_counts.sum() / (num_classes * class_counts)
class_weights_t = torch.tensor(class_weights, dtype=torch.float32, device=device)

criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING, weight=class_weights_t)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = build_warmup_cosine_scheduler(optimizer, EPOCHS, WARMUP_EPOCHS)
early_stopping = EarlyStopping(patience=EARLY_STOPPING_PATIENCE, min_delta=0.001)

best_val_loss = float('inf')
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

print('\nIniciando entrenamiento (Conformer)...')
for epoch in range(EPOCHS):
    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer, device, train=True)
    val_loss, val_acc = run_epoch(model, val_loader, criterion, optimizer, device, train=False)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    lr_actual = optimizer.param_groups[0]['lr']
    print(f'Epoca [{epoch+1}/{EPOCHS}] | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | '
          f'Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}% | LR: {lr_actual:.6f}')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'model_state_dict': model.state_dict(),
            'class_names': dataset.class_names,
            'max_frames': MAX_FRAMES,
            'input_dim': input_dim,
            'val_loss': val_loss,
            'val_acc': val_acc,
        }, VIDEO_MODEL_PATH)
        print(f'  -> Nuevo mejor modelo guardado (Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%)')

    early_stopping(val_loss)
    if early_stopping.early_stop:
        print(f'  -> Early stopping en la epoca {epoch+1}.')
        break

print(f'\nListo. Mejor Val Loss: {best_val_loss:.4f} | Modelo en: {VIDEO_MODEL_PATH}')

## 7. Graficas del modelo de video

In [ ]:
from sklearn.metrics import confusion_matrix

epochs_range = range(1, len(history['train_loss']) + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.plot(epochs_range, history['train_loss'], 'b-', label='Train Loss', linewidth=2)
ax1.plot(epochs_range, history['val_loss'], 'r--', label='Validation Loss', linewidth=2)
ax1.set_title('Convergencia de Perdida (video)')
ax1.set_xlabel('Epocas'); ax1.set_ylabel('Perdida'); ax1.legend(); ax1.grid(True)
ax2.plot(epochs_range, history['train_acc'], 'b-', label='Train Accuracy', linewidth=2)
ax2.plot(epochs_range, history['val_acc'], 'g-', label='Validation Accuracy', linewidth=2)
ax2.set_title('Precision del modelo de video')
ax2.set_xlabel('Epocas'); ax2.set_ylabel('Precision (%)'); ax2.legend(); ax2.grid(True)
plt.tight_layout()
curvas_path = os.path.join(OUTPUTS_DIR, 'landmarks_conformer_curvas.png')
plt.savefig(curvas_path)
plt.show()
print(f'-> Grafico guardado: {curvas_path}')

best_checkpoint = torch.load(VIDEO_MODEL_PATH, map_location=device)
model.load_state_dict(best_checkpoint['model_state_dict'])
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for sequences, labels, lengths in val_loader:
        sequences = sequences.to(device)
        lengths_dev = lengths.to(device)
        mask = make_padding_mask(lengths_dev, sequences.shape[1], device)
        outputs = model(sequences, src_key_padding_mask=mask)
        preds = outputs.argmax(dim=1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())

total = len(all_labels)
top1_acc = 100.0 * sum(p == l for p, l in zip(all_preds, all_labels)) / total
print(f'Top-1 accuracy (val): {top1_acc:.2f}%')

cm = confusion_matrix(all_labels, all_preds, labels=list(range(len(dataset.class_names))))
plt.figure(figsize=(max(8, len(dataset.class_names) * 0.35), max(6, len(dataset.class_names) * 0.35)))
plt.imshow(cm, cmap='Blues')
plt.xticks(range(len(dataset.class_names)), dataset.class_names, rotation=90, fontsize=8)
plt.yticks(range(len(dataset.class_names)), dataset.class_names, fontsize=8)
plt.xlabel('Predicho'); plt.ylabel('Real')
plt.title(f'Matriz de confusion (video, val) -- top1 {top1_acc:.1f}%')
plt.colorbar()
plt.tight_layout()
cm_path = os.path.join(OUTPUTS_DIR, 'matriz_confusion.png')
plt.savefig(cm_path)
plt.show()
print(f'-> Matriz de confusion guardada: {cm_path}')

---
# PARTE B -- Entrenar el modelo de AUDIO (wav2vec2 + clasificador)

## 8. Cargar embeddings (y extraer los que falten)

Usa los embeddings ya calculados en `data/audio_embeddings_cache/` -- si falta alguno (clip nuevo que el notebook de extraccion todavia no proceso), carga wav2vec2 y lo calcula al vuelo, igual que hace `train_audio_classifier.py` en tu PC.

In [ ]:
def find_labeled_clips():
    clips_por_clase = {}
    if not os.path.isdir(RUTA_CLIPS):
        return clips_por_clase
    for clase in sorted(os.listdir(RUTA_CLIPS)):
        clips_dir = os.path.join(RUTA_CLIPS, clase, 'clips')
        if not os.path.isdir(clips_dir):
            continue
        wavs = sorted(os.path.join(clips_dir, f) for f in os.listdir(clips_dir) if f.lower().endswith('.wav'))
        if wavs:
            clips_por_clase[clase] = wavs
    return clips_por_clase


clips_por_clase = find_labeled_clips()
if not clips_por_clase:
    print(f'No encontre clips con audio en {RUTA_CLIPS}.')
else:
    print('Clases encontradas:')
    for clase, wavs in clips_por_clase.items():
        print(f'  {clase}: {len(wavs)} clips')

os.makedirs(AUDIO_EMB_DIR, exist_ok=True)

faltantes = [
    wav_path for wavs in clips_por_clase.values() for wav_path in wavs
    if not os.path.exists(os.path.join(AUDIO_EMB_DIR, os.path.splitext(os.path.basename(wav_path))[0] + '.npy'))
]
print(f'\n{len(faltantes)} clips sin embedding todavia.')

if faltantes:
    from huggingface_hub import hf_hub_download
    from transformers import Wav2Vec2Config, Wav2Vec2FeatureExtractor, Wav2Vec2ForCTC
    import torchaudio

    MODEL_NAME = 'facebook/wav2vec2-xlsr-53-espeak-cv-ft'
    TARGET_SAMPLE_RATE = 16000
    EMBEDDING_DIM = 1024

    print(f'Cargando {MODEL_NAME} para completar embeddings faltantes...')
    feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_NAME)
    config = Wav2Vec2Config.from_pretrained(MODEL_NAME)
    audio_model = Wav2Vec2ForCTC(config)
    weights_path = hf_hub_download(MODEL_NAME, 'pytorch_model.bin')
    state_dict = torch.load(weights_path, map_location='cpu', weights_only=True)
    audio_model.load_state_dict(state_dict)
    audio_model.to(device)
    audio_model.eval()

    def load_audio_16k(wav_path):
        waveform, sr = torchaudio.load(wav_path)
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        if sr != TARGET_SAMPLE_RATE:
            waveform = torchaudio.functional.resample(waveform, sr, TARGET_SAMPLE_RATE)
        return waveform.squeeze(0)

    def extract_embedding(wav_path):
        waveform = load_audio_16k(wav_path)
        inputs = feature_extractor(waveform.numpy(), sampling_rate=TARGET_SAMPLE_RATE, return_tensors='pt')
        with torch.inference_mode():
            hidden = audio_model.wav2vec2(inputs.input_values.to(device)).last_hidden_state
            pooled = hidden.mean(dim=1).squeeze(0)
        return pooled.cpu().numpy()

    for i, wav_path in enumerate(faltantes, 1):
        cache_path = os.path.join(AUDIO_EMB_DIR, os.path.splitext(os.path.basename(wav_path))[0] + '.npy')
        emb = extract_embedding(wav_path)
        np.save(cache_path, emb)
        if i % 20 == 0 or i == len(faltantes):
            print(f'  embeddings nuevos: {i}/{len(faltantes)}')
else:
    EMBEDDING_DIM = 1024
    print('Todos los embeddings ya estaban calculados.')

## 9. Entrenar el clasificador de audio

Guarda `models/audio_classifier.pth` y muestra las graficas de Loss/Accuracy + matriz de confusion al terminar (igual que en tu PC).

In [ ]:
AUDIO_MODEL_PATH = os.path.join(MODELS_DIR, 'audio_classifier.pth')

VAL_FRACTION = 0.2
AUDIO_SEED = 42
AUDIO_EPOCHS = 120
HIDDEN_DIM = 128
DROPOUT = 0.4
AUDIO_WEIGHT_DECAY = 1e-3
AUDIO_LR = 1e-3


class AudioClassifierHead(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_classes, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x):
        return self.net(x)


class_names_audio = sorted(clips_por_clase.keys())
X, y = [], []
for clase in class_names_audio:
    for wav_path in clips_por_clase[clase]:
        cache_path = os.path.join(AUDIO_EMB_DIR, os.path.splitext(os.path.basename(wav_path))[0] + '.npy')
        X.append(np.load(cache_path))
        y.append(class_names_audio.index(clase))
X = np.stack(X).astype(np.float32)
y = np.array(y, dtype=np.int64)
print(f'Total: {len(X)} clips, {len(class_names_audio)} clases: {class_names_audio}')

rng = np.random.RandomState(AUDIO_SEED)
train_idx, val_idx = [], []
for c in np.unique(y):
    idx = np.where(y == c)[0]
    rng.shuffle(idx)
    n_val = max(1, int(round(len(idx) * VAL_FRACTION)))
    val_idx.extend(idx[:n_val])
    train_idx.extend(idx[n_val:])
rng.shuffle(train_idx)
rng.shuffle(val_idx)
train_idx, val_idx = np.array(train_idx), np.array(val_idx)
print(f'Train: {len(train_idx)} | Val: {len(val_idx)}')

mean = X[train_idx].mean(axis=0, keepdims=True)
std = X[train_idx].std(axis=0, keepdims=True) + 1e-6

def to_tensor(idx):
    xs = (X[idx] - mean) / std
    return torch.from_numpy(xs).float().to(device), torch.from_numpy(y[idx]).long().to(device)

x_train, y_train = to_tensor(train_idx)
x_val, y_val = to_tensor(val_idx)

head = AudioClassifierHead(EMBEDDING_DIM, HIDDEN_DIM, len(class_names_audio), DROPOUT).to(device)
audio_optimizer = torch.optim.Adam(head.parameters(), lr=AUDIO_LR, weight_decay=AUDIO_WEIGHT_DECAY)
audio_criterion = nn.CrossEntropyLoss()

audio_history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_acc, best_state = -1.0, None

for epoch in range(1, AUDIO_EPOCHS + 1):
    head.train()
    audio_optimizer.zero_grad()
    logits = head(x_train)
    loss = audio_criterion(logits, y_train)
    loss.backward()
    audio_optimizer.step()

    head.eval()
    with torch.no_grad():
        train_acc = (head(x_train).argmax(1) == y_train).float().mean().item()
        val_logits = head(x_val)
        val_loss = audio_criterion(val_logits, y_val).item()
        val_acc = (val_logits.argmax(1) == y_val).float().mean().item()

    audio_history['train_loss'].append(loss.item())
    audio_history['val_loss'].append(val_loss)
    audio_history['train_acc'].append(train_acc * 100)
    audio_history['val_acc'].append(val_acc * 100)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = {k: v.clone() for k, v in head.state_dict().items()}

    if epoch % 10 == 0 or epoch == 1:
        print(f'  epoch {epoch:3d} | train_loss {loss.item():.3f} | val_loss {val_loss:.3f} | '
              f'train_acc {train_acc*100:.1f}% | val_acc {val_acc*100:.1f}%')

print(f'\nMejor accuracy de validacion: {best_val_acc*100:.2f}%')

torch.save({
    'state_dict': best_state,
    'class_names': class_names_audio,
    'embedding_dim': EMBEDDING_DIM,
    'hidden_dim': HIDDEN_DIM,
    'mean': mean,
    'std': std,
    'val_acc': best_val_acc,
}, AUDIO_MODEL_PATH)
print(f'Guardado en {AUDIO_MODEL_PATH}')

## 10. Graficas del modelo de audio

In [ ]:
epochs_range = range(1, AUDIO_EPOCHS + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.plot(epochs_range, audio_history['train_loss'], 'b-', label='Train Loss', linewidth=2)
ax1.plot(epochs_range, audio_history['val_loss'], 'r--', label='Validation Loss', linewidth=2)
ax1.set_title('Perdida (audio)')
ax1.set_xlabel('Epocas'); ax1.set_ylabel('Perdida'); ax1.legend(); ax1.grid(True)
ax2.plot(epochs_range, audio_history['train_acc'], 'b-', label='Train Accuracy', linewidth=2)
ax2.plot(epochs_range, audio_history['val_acc'], 'g-', label='Validation Accuracy', linewidth=2)
ax2.set_title('Precision del clasificador de audio')
ax2.set_xlabel('Epocas'); ax2.set_ylabel('Precision (%)'); ax2.legend(); ax2.grid(True)
plt.tight_layout()
curvas_audio_path = os.path.join(OUTPUTS_DIR, 'audio_classifier_curvas.png')
plt.savefig(curvas_audio_path)
plt.show()
print(f'-> Grafico guardado: {curvas_audio_path}')

head.load_state_dict(best_state)
head.eval()
with torch.no_grad():
    val_pred = head(x_val).argmax(1).cpu().numpy()
val_true = y[val_idx]
n = len(class_names_audio)
conf = np.zeros((n, n), dtype=int)
for t, p in zip(val_true, val_pred):
    conf[t, p] += 1

fig, ax = plt.subplots(figsize=(max(6, n * 0.45), max(5, n * 0.45)))
im = ax.imshow(conf, cmap='Blues')
ax.set_xticks(range(n)); ax.set_xticklabels(class_names_audio, rotation=90, fontsize=8)
ax.set_yticks(range(n)); ax.set_yticklabels(class_names_audio, fontsize=8)
ax.set_xlabel('Predicho'); ax.set_ylabel('Real')
ax.set_title(f'Matriz de confusion (audio) -- val acc {best_val_acc*100:.1f}%')
plt.colorbar(im)
plt.tight_layout()
cm_audio_path = os.path.join(OUTPUTS_DIR, 'audio_matriz_confusion.png')
plt.savefig(cm_audio_path)
plt.show()
print(f'-> Matriz de confusion guardada: {cm_audio_path}')

## Listo

Modelos entrenados guardados en tu Drive:

- `models/best.pth` -- modelo visual (video).
- `models/audio_classifier.pth` -- modelo de audio.

Bajalos de Drive y ponelos en la carpeta `models/` de tu proyecto local para usarlos con `server/main.py` y `scripts/probar_modelo.py`.